# UNIVERSIDAD DE CARABOBO
## Asignación 2: Data Preparation y Feature Engineering
### Predictor de victorias de Super Smash Bros. Melee

---

## Sección 1 — Propósito y Contexto del Problema

Este notebook es el entregable central de la Asignación 2: convierte el estado
bruto de las partidas en una matriz numérica lista para modelado, aplicando
feature engineering y los criterios de prevención de *data leakage* que exige la
rúbrica.

**Unidad de observación.** Cada fila del dataset representa **un frame
individual** de una partida 1v1 de Super Smash Bros. Melee. El juego corre a 60
frames por segundo, de modo que cada fila es una instantánea del estado completo
de los dos jugadores en 1/60 s de tiempo real: posición, daño, stocks, escudos,
acciones en curso y velocidades. El dataset tiene **540 940 frames** repartidos
en **49 partidas**.

**Variable objetivo.** `outcome` vale **1 si gana el Jugador 1** y **2 si gana el
Jugador 2**. Está definida a nivel de *partida*, no de frame: todos los frames
de una misma partida comparten siempre el mismo valor de `outcome`.

**Qué significa predecir.** Una predicción es el ganador de la partida deducido a
partir del estado de **un único frame**. En el momento de emitirla, lo único
disponible es el estado de ese frame concreto. **No** está disponible ningún frame
posterior de la partida, ni el `outcome`, ni el `match_id`: los frames llegan
desordenados dentro de cada archivo y el `match_id` se reconstruye agrupando
secuencias de frames contiguos, lo que exige mirar también los frames futuros.
Por eso `match_id` es legítimo para particionar los datos, pero nunca puede
entrar como característica.

**Variables que causan data leakage y quedan excluidas del entrenamiento:**

| Variable | Motivo de la exclusión |
|---|---|
| `row_id` | Identificador único de cada frame. Es un valor arbitrario sin relación con el estado del juego; el modelo lo memorizaría y sobreajustaría. |
| `match_id` | Identificador del grupo (la partida) al que pertenece el frame. Es la clave de agrupamiento: incluirla permite reconocer la partida y memorizar su resultado, que es literalmente la variable objetivo. |

`outcome` se emplea **exclusivamente** como vector objetivo del ajuste, nunca como
entrada. Las tres quedan declaradas en `EXCLUDE_COLS` más adelante.

## Sección 2 — Hallazgos Relevantes del EDA

Resumen de lo observado y cuantificado en `notebooks/01_eda.ipynb`, sobre las
mismas 540 940 filas que se cargan aquí.

**1. Volumen y agrupamiento.** 540 940 frames distribuidos en **49 partidas
reconstruidas** (25 en la partición `0000` y 24 en la `0001`), con una mediana de
10 627 frames por partida, equivalente a unos 2,95 minutos de juego real.

**2. Balance de clases.** El Jugador 1 gana el **55,4459 %** de los frames
(299 929) y el Jugador 2 el **44,5541 %** (241 011). El balance es razonable y
no requiere reponderación, pero conviene recordar que la fiabilidad de la
heurística de dominio crece con el progreso de la partida: es del 71,25 % en los
primeros 5 000 frames y del 92,21 % a partir de 12 000.

**3. Calidad de los datos: ausencia total de nulos.** El parquet tiene **cero
valores nulos en las 47 columnas**. Esto es deliberado y no accidental: el pipeline de extracción preserva la
ausencia real de información en lugar de rellenarla con valores por defecto. Aun así se mantienen imputaciones en el
pipeline (§9) como red de seguridad para columnas que en producción podrían
llegar incompletas.

**4. Fuerte correlación espacial y física.** Las variables de posición y estado
físico están fuertemente correlacionadas entre sí: `p1_x` recorre
[-255,89 ; 273,39], `p1_y` [-149,09 ; 203,59], las velocidades horizontales
±18,72 y el daño acumulado 0–213 %. Además se detectaron **338 estados duplicados
físicos** (0,0625 % del total) y las 676 filas implicadas tienen `frame < 0`: son
fotogramas del *countdown* previo al combate, con los personajes congelados antes
de que se habilite el control.

## Sección 3 — Tabla EDA → Decisiones de Preparación

Cada hallazgo del EDA se traduce en una decisión concreta del pipeline. La columna
**Evidencia** recoge el número medido sobre el dataset, no una impresión
cualitativa.

| Hallazgo del EDA | Evidencia | Decisión de preparación | Justificación |
|---|---|---|---|
| **a)** Los frames están agrupados en partidas completas y son temporalmente autocorrelacionados | 540 940 filas colapsan en solo 49 `match_id`; los frames contiguos de una partida comparten estado y resultado | **Split agrupado** con `GroupShuffleSplit` sobre `match_id` (§6) | Un split aleatorio por fila reparte frames de la misma partida entre train y test; el modelo vería la misma partida en ambos lados y el accuracy en test mediría memorización, no generalización |
| **b)** La ventaja en combate es *relativa*: depende del contraste entre ambos jugadores | `stock_diff = p1_stock - p2_stock` correlaciona **-0,4588** con `outcome`; `percent_diff = p2_percent - p1_percent` correlaciona **-0,0643** | Crear **`stock_diff`** y **`percent_diff`** (§7) | Los stocks y el daño absolutos no son comparables entre jugadores; la diferencia es interpretable y encapsula la ventaja de forma directa |
| **c)** Estar fuera del escenario es un estado peligroso y asimétrico | `off_stage` es booleano; **63 106 filas (11,67 %)** tienen a P2 fuera y P1 dentro | Crear el flag **`p1_edgeguard_advantage`** (§7) | Regla táctica asimétrica: solo hay ventaja si el defensor está fuera y el atacante no. Exige combinar dos booleanos; leer `off_stage` en crudo no la captura |
| **d)** Personaje y stage son categorías nominales de alta relevancia | 6 stages, 14 personajes de P1 y 12 de P2 en el fold de train | **`OneHotEncoder(handle_unknown='ignore')`** (§9) | Son variables nominales sin orden intrínseco. `handle_unknown='ignore'` evita que el pipeline se rompa si en producción aparece un personaje no visto en train |
| **e)** Las variables numéricas viven en escalas muy distintas | `p1_x` ∈ [-255,89 ; 273,39] frente a `p1_percent` ∈ [0 ; 213]; `stock_ratio` ∈ [0,25 ; 4,00] | **`StandardScaler`** tras `SimpleImputer(median)` (§9) | Sin estandarizar, distancias y coeficientes quedan dominados por las variables de mayor magnitud y el optimizador converge de forma sesgada hacia las coordenadas |
| **f)** Existen frames de cuenta regresiva, con los personajes congelados | **5 904 frames** con `frame < 0` (1,0914 % del total) | **Descartar** en la celda de depuración, antes de la §6 | Antes de que se habilite el control no hay acción ni movimiento que modelar: la señal es constante y solo diluye el resto de las señales |
| **g)** Un stock en 0 significa que la partida ya está decidida | **48 frames** (0,0089 %) con `p1_stock == 0` o `p2_stock == 0` | **Descartar** en la celda de depuración | El resultado ya está determinado cuando alguien agota sus stocks, así que el frame no informa quién ganó: incluirlo filtra la respuesta directamente hacia el objetivo |
| **h)** La transformación Zelda/Sheik parte una partida real en dos segmentos | `0000_m16` (14 805 filas, frames -123..14681, SHEIK vs MARTH) y `0000_m25` (177 filas, frames 14682..14858, ZELDA vs MARTH) son la misma partida de 14 982 frames. **Verificado sobre el parquet:** comparten archivo, stage (BATTLEFIELD) y trajes (p1=0, p2=3), y `f_min(m25) = 14682 = f_max(m16) + 1`. Es el único par de los 49 con esa propiedad: de 5 pares con stage y trajes idénticos, 4 no son contiguos. La única diferencia entre los dos grupos es `p1_character`. En el dataset ya depurado quedan 14 858 filas (14 682 + 176). | **Documentar como limitación** y exigir con `assert` que ambos fragmentos caigan del mismo lado del split | `assign_match_ids` agrupa por `(stage, p1_costume, p2_costume, p1_character, p2_character)` y la transformación cambia `p1_character` de SHEIK a ZELDA a mitad de partida. Ambos segmentos comparten `outcome`, de modo que separarlos en el split reintroduciría leakage. Conviene notar que esto es una inferencia a partir del agrupamiento, no un identificador de partida que venga en los datos |

> **Nota sobre `stock_ratio`.** Su denominador incorpora `EPSILON = 1e-5` para
> evitar divisiones por cero, pero tras la depuración de la fila **(g)** el
> `p2_stock` mínimo es 1, así que el denominador es siempre mayor o igual que 1 y
> la variable queda acotada en **[0,2500 ; 4,0000]**, con máximo **3,99996**.
> `EPSILON` deja de ser determinante, `StandardScaler` ya no comprime esta variable
> y las 26 filas que alcanzaban 400 000 ya no forman parte del dataset.

## Sección 4 — Imports, Constantes y Rutas
---

In [1]:
# --- Librería estándar -----------------------------------------------------
import os
import sys
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Set

# --- Librerías de terceros -------------------------------------------------
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    StandardScaler,
)

# --- Verificación de versión ----------------------------------------------
# Debe ejecutarse ANTES del import de sklearn.utils: Tags y TransformerTags
# no existen en scikit-learn < 1.6, de modo que importarlos primero fallaría
# con ImportError antes de que el assert pudiera evaluarse. El resto de
# imports de sklearn de arriba son seguros en cualquier versión.
import sklearn

_version = tuple(int(p) for p in sklearn.__version__.split(".")[:2])
assert _version >= (1, 6), (
    f"Se requiere scikit-learn >= 1.6 (instalado {sklearn.__version__}): "
    "Tags y TransformerTags son obligatorios para MeleeFeatureEngineer."
)

from sklearn.utils import Tags, TransformerTags  # noqa: E402

# --- Constantes ------------------------------------------------------------
RANDOM_STATE = 42
TEST_SIZE = 0.20
EPSILON = 1e-5

# --- Rutas de datos --------------------------------------------------------
# El notebook funciona desde la raíz del repositorio, desde notebooks/ y desde
# un entorno donde el parquet se copia junto al notebook (Colab, entrega
# autocontenida): se prueban las tres ubicaciones y se usa la primera disponible.
CANDIDATOS_DATOS = (Path("../data"), Path("data"), Path("."))
DATA_PATH = next(
    (base / "processed_sample.parquet" for base in CANDIDATOS_DATOS
     if (base / "processed_sample.parquet").exists()),
    CANDIDATOS_DATOS[0] / "processed_sample.parquet",
)

print(f"Intérprete       : {Path(sys.executable).name} "
      f"(Python {sys.version_info.major}.{sys.version_info.minor})")
print(f"numpy            : {np.__version__}")
print(f"pandas           : {pd.__version__}")
print(f"scikit-learn     : {sklearn.__version__}")

print("\nEntorno verificado.")

Intérprete       : python (Python 3.14)
numpy            : 2.5.3
pandas           : 3.0.6
scikit-learn     : 1.9.1

Entorno verificado.


**Interpretación.** El notebook se autoverifica: imprime el intérprete real que
ejecuta las celdas y las versiones de `numpy`, `pandas` y `scikit-learn`. Esto
importa porque un entorno mal activado produce un `ModuleNotFoundError` en la
celda siguiente, difícil de diagnosticar si no se sabe qué intérprete está en
juego. Al ser el notebook autocontenido, esta celda se limita a imprimir el
intérprete y las versiones de las librerías: no necesita resolver la raíz del
proyecto ni importar ningún módulo local.

## Sección 5 — Carga y Validación de Datos

---

El dataset puede entrar por **dos rutas**, y el notebook resuelve cuál usar antes
de leer nada:

1. **Parquet.** Si existe `data/processed_sample.parquet` (o junto al notebook),
   se lee directamente. Es la ruta rápida y la que se usa en el desarrollo de
   este notebook.
2. **Generación desde los datos crudos.** Si no existe el parquet, se buscan
   `train/0000.msgpack.zst`, `train/0001.msgpack.zst` y `training_labels.csv` en
   cuatro ubicaciones candidatas, y el DataFrame se reconstruye **en memoria**,
   sin escribir ningún archivo.

Ambas rutas producen exactamente el mismo DataFrame de 540 940 filas y 47
columnas. La ruta cruda no necesita el parquet previo, pero sí las bibliotecas
`msgpack` y `zstandard`, que solo se exigen cuando se usa esa ruta: si el parquet
está disponible, el notebook funciona sin ellas.

El esquema, la forma esperada y las aserciones de las subsecciones siguientes se
aplican igual sobre las dos rutas, de modo que un dataset mal construido falla
igual en cualquiera de los dos casos.

In [2]:
# --- Dependencias opcionales para leer los datos crudos --------------------
# El import NO es fatal. Si el parquet ya existe, el notebook no necesita estas
# librerías en absoluto; solo son obligatorias en la ruta de generación desde los
# .msgpack.zst, y build_dataset_from_raw lanza un ImportError claro si faltan.
# El import queda a nivel de celda porque las funciones copiadas más abajo usan
# `zstd` y `msgpack` como nombres globales y el notebook comparte un único
# namespace entre celdas.
try:
    import msgpack
    import zstandard as zstd
except ImportError:  # depende del entorno; no es un error aquí
    msgpack = None
    zstd = None


def load_labels_subset(labels_csv_path: str, target_file_ids: Set[str]) -> Dict[str, int]:
    """
    Streams the training labels CSV file line-by-line and loads outcomes only
    for the specified file IDs to avoid loading the entire 880+ MB file into memory.

    Args:
        labels_csv_path: Path to `training_labels.csv`.
        target_file_ids: Set of file prefixes to extract (e.g., {'0000', '0001'}).

    Returns:
        Dictionary mapping row_id (e.g. '0000_123') -> outcome (1 or 2).
    """
    labels = {}
    if not os.path.exists(labels_csv_path):
        print(f"[Warning] Labels file not found at: {labels_csv_path}")
        return labels

    print(f"Loading labels for file prefixes: {target_file_ids}...")
    max_prefix = max(target_file_ids) if target_file_ids else ""
    with open(labels_csv_path, "r", encoding="utf-8") as f:
        _header = f.readline()
        for line in f:
            line_clean = line.strip()
            if not line_clean:
                continue
            parts = line_clean.split(",")
            if len(parts) == 2:
                row_id, outcome = parts
                prefix = row_id.split("_")[0]
                if prefix in target_file_ids:
                    labels[row_id] = int(outcome)
                elif max_prefix and prefix > max_prefix:
                    # Labels are sorted by prefix; stop once we exceed the highest target prefix
                    break
    print(f"Loaded {len(labels):,} label records.")
    return labels


def assign_match_ids(frames_list: list, file_id: str) -> Dict[int, str]:
    """
    Reconstructs distinct matches from a shuffled list of frames.
    
    Frames are grouped by stage, player costumes, and characters, then sorted
    chronologically by frame index. Any sequence where consecutive frames increment
    by 1 belongs to the exact same match.
    
    Args:
        frames_list: Unpacked list of frame dictionaries from a single archive.
        file_id: The partition identifier (e.g. "0000").
        
    Returns:
        Dictionary mapping original frame index in the archive -> match_id.
    """
    groups = defaultdict(list)
    for idx, fr in enumerate(frames_list):
        p1 = fr.get("players", {}).get(1, {})
        p2 = fr.get("players", {}).get(2, {})
        key = (
            fr.get("stage"),
            p1.get("costume"),
            p2.get("costume"),
            p1.get("character"),
            p2.get("character")
        )
        groups[key].append((fr.get("frame"), idx))

    match_id_by_idx = {}
    match_counter = 0
    for _key, items in groups.items():
        items.sort(key=lambda x: x[0])
        current_match = [items[0]]
        for i in range(1, len(items)):
            if items[i][0] == items[i - 1][0] + 1:
                current_match.append(items[i])
            else:
                match_counter += 1
                mid = f"{file_id}_m{match_counter:02d}"
                for _, orig_idx in current_match:
                    match_id_by_idx[orig_idx] = mid
                current_match = [items[i]]
        if current_match:
            match_counter += 1
            mid = f"{file_id}_m{match_counter:02d}"
            for _, orig_idx in current_match:
                match_id_by_idx[orig_idx] = mid

    return match_id_by_idx


def flatten_frame(frame_data: dict, row_id: str, match_id: str, outcome: Optional[int] = None) -> dict:
    """
    Flattens a single game frame dictionary into a flat record.
    Preserves None for absent fields rather than masking them with artificial defaults.

    Args:
        frame_data: Raw unpacked dictionary representing a frame.
        row_id: Identifier corresponding to the file and frame index (e.g., '0000_0').
        match_id: Reconstructed identifier of the match (e.g. '0000_m01').
        outcome: Match winner outcome (1, 2, or None if test set).

    Returns:
        Dictionary with flattened column attributes.
    """
    players = frame_data.get("players", {})
    p1 = players.get(1, {})
    p2 = players.get(2, {})

    p1_pos = p1.get("position", {}) if isinstance(p1.get("position"), dict) else {}
    p2_pos = p2.get("position", {}) if isinstance(p2.get("position"), dict) else {}

    record = {
        # Identifiers and global match context
        "row_id": row_id,
        "match_id": match_id,
        "frame": frame_data.get("frame"),
        "stage": frame_data.get("stage"),
        
        # Player 1 Attributes (no masking defaults)
        "p1_character": p1.get("character"),
        "p1_costume": p1.get("costume"),
        "p1_stock": p1.get("stock"),
        "p1_percent": p1.get("percent"),
        "p1_shield_strength": p1.get("shield_strength"),
        "p1_is_powershield": p1.get("is_powershield"),
        "p1_facing": p1.get("facing"),
        "p1_action": p1.get("action"),
        "p1_action_frame": p1.get("action_frame"),
        "p1_invulnerable": p1.get("invulnerable"),
        "p1_invulnerability_left": p1.get("invulnerability_left"),
        "p1_hitlag_left": p1.get("hitlag_left"),
        "p1_hitstun_frames_left": p1.get("hitstun_frames_left"),
        "p1_jumps_left": p1.get("jumps_left"),
        "p1_on_ground": p1.get("on_ground"),
        "p1_off_stage": p1.get("off_stage"),
        "p1_x": p1_pos.get("x"),
        "p1_y": p1_pos.get("y"),
        "p1_speed_air_x_self": p1.get("speed_air_x_self"),
        "p1_speed_y_self": p1.get("speed_y_self"),
        "p1_speed_ground_x_self": p1.get("speed_ground_x_self"),

        # Player 2 Attributes (no masking defaults)
        "p2_character": p2.get("character"),
        "p2_costume": p2.get("costume"),
        "p2_stock": p2.get("stock"),
        "p2_percent": p2.get("percent"),
        "p2_shield_strength": p2.get("shield_strength"),
        "p2_is_powershield": p2.get("is_powershield"),
        "p2_facing": p2.get("facing"),
        "p2_action": p2.get("action"),
        "p2_action_frame": p2.get("action_frame"),
        "p2_invulnerable": p2.get("invulnerable"),
        "p2_invulnerability_left": p2.get("invulnerability_left"),
        "p2_hitlag_left": p2.get("hitlag_left"),
        "p2_hitstun_frames_left": p2.get("hitstun_frames_left"),
        "p2_jumps_left": p2.get("jumps_left"),
        "p2_on_ground": p2.get("on_ground"),
        "p2_off_stage": p2.get("off_stage"),
        "p2_x": p2_pos.get("x"),
        "p2_y": p2_pos.get("y"),
        "p2_speed_air_x_self": p2.get("speed_air_x_self"),
        "p2_speed_y_self": p2.get("speed_y_self"),
        "p2_speed_ground_x_self": p2.get("speed_ground_x_self"),

        # Target outcome (1: Player 1 won, 2: Player 2 won)
        "outcome": outcome
    }
    return record


def extract_and_flatten_file(
    file_path: str,
    labels_dict: Optional[Dict[str, int]] = None,
    max_frames: Optional[int] = None
) -> pd.DataFrame:
    """
    Decompresses and unpacks a .msgpack.zst file, reconstructs matches, and flattens all frames.

    Args:
        file_path: Path to the .msgpack.zst file.
        labels_dict: Optional lookup dictionary of row_id -> outcome.
        max_frames: Optional maximum number of frames to extract.

    Returns:
        Pandas DataFrame containing flattened observations with match_id.
    """
    file_path_obj = Path(file_path)
    file_id = file_path_obj.name.split(".")[0]  # e.g., "0000"

    print(f"Processing '{file_path_obj.name}'...")
    dctx = zstd.ZstdDecompressor()
    records = []

    with open(file_path, "rb") as compressed_file:
        with dctx.stream_reader(compressed_file) as reader:
            unpacker = msgpack.Unpacker(reader, raw=False, strict_map_key=False)
            for raw_content in unpacker:
                if isinstance(raw_content, list):
                    match_map = assign_match_ids(raw_content, file_id)
                    for idx, frame_dict in enumerate(raw_content):
                        if max_frames is not None and len(records) >= max_frames:
                            break
                        row_id = f"{file_id}_{idx}"
                        match_id = match_map.get(idx, f"{file_id}_m00")
                        outcome = labels_dict.get(row_id) if labels_dict else None
                        records.append(flatten_frame(frame_dict, row_id, match_id, outcome))
                if max_frames is not None and len(records) >= max_frames:
                    break

    df = pd.DataFrame(records)
    print(f"Extracted {len(df):,} rows and {df['match_id'].nunique()} matches from '{file_path_obj.name}'.")
    return df


def build_dataset_from_raw(raw_dir: Path, file_ids: Sequence[str]) -> pd.DataFrame:
    """Construye el dataset aplanado desde los archivos .msgpack.zst.

    Reúne los frames de las particiones indicadas, les asigna un `match_id`
    reconstruido y añade el `outcome` leído de `training_labels.csv`. El
    resultado se devuelve **solo en memoria**: no escribe ningún archivo.

    Parameters
    ----------
    raw_dir : pathlib.Path
        Directorio que contiene `train/<file_id>.msgpack.zst` y
        `training_labels.csv`.
    file_ids : sequence of str
        Identificadores de partición, por ejemplo ``["0000", "0001"]``.

    Returns
    -------
    pandas.DataFrame
        DataFrame concatenado con índice reiniciado.

    Raises
    ------
    ImportError
        Si `msgpack` o `zstandard` no están disponibles en el entorno.
    """
    faltantes = [
        nombre for nombre, modulo in (("msgpack", msgpack), ("zstandard", zstd))
        if modulo is None
    ]
    if faltantes:
        raise ImportError(
            "Para generar el dataset desde los archivos .msgpack.zst hacen falta "
            f"las bibliotecas {', '.join(faltantes)}, que no están disponibles "
            "en este entorno. Sin ellas no se puede leer el dataset crudo; la "
            "alternativa es disponer de data/processed_sample.parquet en data/."
        )

    base = Path(raw_dir)
    files = [base / "train" / f"{file_id}.msgpack.zst" for file_id in file_ids]
    labels_csv = base / "training_labels.csv"
    ids = {path.name.split(".")[0] for path in files}

    etiquetas = load_labels_subset(str(labels_csv), ids)
    bloques = [
        extract_and_flatten_file(str(path), labels_dict=etiquetas)
        for path in files
    ]
    return pd.concat(bloques, ignore_index=True)

In [3]:
# --- Carga -----------------------------------------------------------------
# Dos rutas de entrada. Si existe el parquet se lee directamente; si no, se
# genera el DataFrame en memoria desde los .msgpack.zst del dataset crudo. Las
# dos producen exactamente el mismo DataFrame, sin escribir ningún archivo.
BASES_CRUDOS = (
    Path("../data/archive"), Path("data/archive"),
    Path("../archive"), Path("archive"),
)
ARCHIVOS_CRUDOS = (
    "train/0000.msgpack.zst", "train/0001.msgpack.zst", "training_labels.csv",
)

df = None
origen = None

if DATA_PATH.exists():
    df = pd.read_parquet(DATA_PATH)
    origen = f"parquet en {DATA_PATH}"
else:
    for base in BASES_CRUDOS:
        if all((base / relativo).exists() for relativo in ARCHIVOS_CRUDOS):
            df = build_dataset_from_raw(base, ["0000", "0001"])
            origen = f"generado en memoria desde {base} ({', '.join(ARCHIVOS_CRUDOS)})"
            break

if df is None:
    raise FileNotFoundError(
        "No se encontró data/processed_sample.parquet ni los archivos crudos "
        "del dataset de SSBM.\n"
        f"  Parquet buscado en: {', '.join(str(b) for b in CANDIDATOS_DATOS)}\n"
        f"  Crudos buscados ({', '.join(ARCHIVOS_CRUDOS)}) en: "
        f"{', '.join(str(b) for b in BASES_CRUDOS)}"
    )

EXPECTED_SHAPE = (540_940, 47)
TARGET = "outcome"
ID_FRAME = "row_id"
ID_GROUP = "match_id"

print(f"Origen de los datos : {origen}")
print(f"Shape         : {df.shape}")

# --- Validación de shape ---------------------------------------------------
assert df.shape == EXPECTED_SHAPE, f"Shape inesperado: {df.shape} != {EXPECTED_SHAPE}"
print(f"Shape esperado: {EXPECTED_SHAPE}  ->  OK")

# --- Validación de schema --------------------------------------------------
# Nota: en pandas 3.x el dtype de las columnas de texto es StringDtype, que NO
# satisface la comparación dtype == str. Por eso se usa is_string_dtype.
SCHEMA = {
    ID_FRAME: "string",
    ID_GROUP: "string",
    "frame": "integer",
    "stage": "string",
    TARGET: "integer",
    "p1_character": "string",
    "p2_character": "string",
    "p1_stock": "integer",
    "p2_stock": "integer",
    "p1_percent": "integer",
    "p2_percent": "integer",
    "p1_off_stage": "boolean",
    "p2_off_stage": "boolean",
    "p1_x": "float",
    "p2_x": "float",
}

fallos = []
for columna, esperado in SCHEMA.items():
    if columna not in df.columns:
        fallos.append(f"{columna}: ausente")
        continue
    serie = df[columna]
    if esperado == "string":
        ok = pd.api.types.is_string_dtype(serie) or serie.dtype == object
    elif esperado == "integer":
        ok = pd.api.types.is_integer_dtype(serie)
    elif esperado == "float":
        ok = pd.api.types.is_float_dtype(serie)
    else:
        ok = pd.api.types.is_bool_dtype(serie)
    if not ok:
        fallos.append(f"{columna}: {serie.dtype} != {esperado}")

print(f"\nValidación de schema: "
      f"{'OK, las ' + str(len(SCHEMA)) + ' columnas coinciden' if not fallos else fallos}")
assert not fallos, fallos

# --- Validación del target -------------------------------------------------
nulos_target = int(df[TARGET].isna().sum())
valores_target = sorted(df[TARGET].unique().tolist())
print(f"\nTarget '{TARGET}': {nulos_target} nulos, valores {valores_target}")
assert nulos_target == 0, "El target no puede contener valores nulos."
assert valores_target == [1, 2], "El target debe contener exactamente {1, 2}."

# --- Unicidad de claves ----------------------------------------------------
unicos_row = int(df[ID_FRAME].nunique())
partidas = int(df[ID_GROUP].nunique())
print(f"\n'{ID_FRAME}' únicos: {unicos_row:,} de {len(df):,} filas "
      f"({'100 % únicos' if unicos_row == len(df) else 'HAY DUPLICADOS'})")
print(f"'{ID_GROUP}' únicos: {partidas} partidas")
assert unicos_row == len(df), "row_id debe ser único en el dataset."
assert TARGET in df.columns and ID_GROUP in df.columns

print(f"\nNulos totales en el dataset: {int(df.isna().sum().sum())}")
print("Datos cargados y validados.")

Origen de los datos : parquet en ../data/processed_sample.parquet
Shape         : (540940, 47)
Shape esperado: (540940, 47)  ->  OK

Validación de schema: OK, las 15 columnas coinciden

Target 'outcome': 0 nulos, valores [1, 2]

'row_id' únicos: 540,940 de 540,940 filas (100 % únicos)
'match_id' únicos: 49 partidas

Nulos totales en el dataset: 0
Datos cargados y validados.


**Interpretación.** Las cuatro validaciones son compuertas (`assert`), no
impresiones: si el dataset cambia de forma, el notebook se detiene aquí en lugar
de producir un pipeline silenciosamente incorrecto. La validación del schema usa
`pd.api.types.is_string_dtype()` porque en `pandas` 3.x el dtype de las columnas
de texto es `StringDtype`, que no satisface la comparación directa con `str`.

El resultado confirma los 540 940 frames, las 47 columnas, las 49 partidas,
`row_id` 100 % único y cero nulos en el conjunto completo.

In [4]:
# --- Depuración de frames sin señal de combate ---------------------------
# Dos tipos de frames no aportan información sobre el ganador:
#   1. frame < 0                     -> countdown previo al combate
#   2. p1_stock == 0 o p2_stock == 0 -> la partida ya está decidida
# Es una regla fija, no aprendida de los datos: se aplica por igual a todas las
# filas y antes de la partición, de modo que train y test reciben el mismo
# criterio. Se filtra con una máscara booleana y se reasigna df, sin modificar
# el objeto original.
countdown_mask = df["frame"] < 0
decided_mask = (df["p1_stock"] == 0) | (df["p2_stock"] == 0)
discard_mask = countdown_mask | decided_mask

rows_before = len(df)
discarded = int(discard_mask.sum())

df = df.loc[~discard_mask].reset_index(drop=True)

print(f"Filas antes del filtrado : {rows_before:,}")
print(f"  frame < 0 (countdown)   : {int(countdown_mask.sum()):,} ({countdown_mask.mean():.4%})")
print(f"  stock == 0 (decidida)   : {int(decided_mask.sum()):,} ({decided_mask.mean():.4%})")
print(f"Filas descartadas        : {discarded:,} ({discarded / rows_before:.4%})")
print(f"Filas conservadas        : {len(df):,} ({len(df) / rows_before:.4%})")
print(f"Partidas                 : {df[ID_GROUP].nunique()}")

assert not (df["frame"] < 0).any(), "Quedan frames de countdown."
assert (df["p1_stock"] > 0).all() and (df["p2_stock"] > 0).all(), "Quedan frames con stock 0."
print("Depuración completada.")

Filas antes del filtrado : 540,940
  frame < 0 (countdown)   : 5,904 (1.0914%)
  stock == 0 (decidida)   : 48 (0.0089%)
Filas descartadas        : 5,952 (1.1003%)
Filas conservadas        : 534,988 (98.8997%)
Partidas                 : 49
Depuración completada.


## Sección 6 — Separación Train / Test (Prevención Total de Leakage)
---

In [5]:
# --- Separación ANTES de ajustar cualquier transformer ---------------------
# El orden es lo importante: si el ColumnTransformer se ajustara sobre el
# dataset completo, su mediana y su media se calcularían también con los frames
# de test. Eso es fuga de información a través de las estadísticas de
# preprocesado, y ocurre en silencio.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)
train_idx, test_idx = next(splitter.split(df, df[TARGET], groups=df[ID_GROUP]))

X_train = df.iloc[train_idx].drop(columns=[TARGET])
X_test = df.iloc[test_idx].drop(columns=[TARGET])
y_train = df[TARGET].iloc[train_idx]
y_test = df[TARGET].iloc[test_idx]

partidas_train = set(df[ID_GROUP].iloc[train_idx])
partidas_test = set(df[ID_GROUP].iloc[test_idx])

print(f"Train: {len(X_train):,} filas | {len(partidas_train)} partidas")
print(f"Test : {len(X_test):,} filas | {len(partidas_test)} partidas")
print(f"Proporción de test: {len(X_test) / len(df):.4f}")

# --- Verificación de no solapamiento ---------------------------------------
solapamiento = partidas_train.intersection(partidas_test)
print(f"\nset(train_matches).intersection(test_matches) == set() -> {solapamiento == set()}")
print(f"Partidas presentes en ambos conjuntos: {len(solapamiento)}")
assert solapamiento == set(), "Hay partidas compartidas entre train y test."

# --- Distribución del objetivo en ambos conjuntos --------------------------
print(f"\nBalance train: P1 {float((y_train == 1).mean()):.4%} | "
      f"P2 {float((y_train == 2).mean()):.4%}")
print(f"Balance test : P1 {float((y_test == 1).mean()):.4%} | "
      f"P2 {float((y_test == 2).mean()):.4%}")
print("\nSplit agrupado completado.")

Train: 426,378 filas | 39 partidas
Test : 108,610 filas | 10 partidas
Proporción de test: 0.2030

set(train_matches).intersection(test_matches) == set() -> True
Partidas presentes en ambos conjuntos: 0

Balance train: P1 55.2847% | P2 44.7153%
Balance test : P1 56.1486% | P2 43.8514%

Split agrupado completado.


In [6]:
# --- Comprobación de partida fragmentada ----------------------------------
# 0000_m16 y 0000_m25 son dos segmentos de la MISMA partida real: la
# transformación de Zelda cambia p1_character de SHEIK a ZELDA a mitad de
# partida y eso rompe el agrupamiento en dos match_id distintos.
# GroupShuffleSplit los trata como grupos independientes, así que podrían caer
# en lados opuestos; como ambos comparten outcome, separarlos filtraría la
# respuesta del test.
PAR_FRAGMENTADO = ("0000_m16", "0000_m25")
lados = {m in partidas_test for m in PAR_FRAGMENTADO}
assert len(lados) == 1, "La partida Sheik/Zelda quedó repartida entre train y test."

_lado = "test" if PAR_FRAGMENTADO[0] in partidas_test else "train"
print(f"Comprobación de partida fragmentada: OK "
      f"(m16 y m25 ambos en {_lado})")

Comprobación de partida fragmentada: OK (m16 y m25 ambos en train)


**Interpretación — por qué el leakage aquí es masivo.** Un split aleatorio por
fila es indistinguible de partir cada partida en trozos: con 534 988 frames de
autocorrelación temporal y solo 49 grupos, es prácticamente seguro que **cada**
partida aparezca simultáneamente en train y en test. Como las filas de una misma
partida comparten el `outcome`, un modelo podría acercarse al 100 % de accuracy
en test sin haber aprendido nada del juego: bastaría con reconocer el estado
típico de esa partida concreta. El EDA ya lo anticipa: la fiabilidad de la heurística de dominio sube del
71,25 % en los primeros 5 000 frames al 92,21 % al final de la partida. Eso
sugiere que el progreso de la partida es una señal fuerte, pero el EDA no separó
ese efecto del efecto de reconocer la partida concreta, así que no permite
concluir que la señal esté en *qué partida es*. El split agrupado es precisamente
lo que permitiría distinguir ambas cosas.

`GroupShuffleSplit` resuelve el problema a nivel de grupo: las 39 partidas de
train y las 10 de test son conjuntos **disjuntos**, como demuestra la intersección
vacía. El coste es que, con solo 49 grupos, la granularidad del split es gruesa
(una partida entera se mueve al test), de modo que el accuracy de test tendrá
más varianza que con un split por filas. Es el precio correcto: preferimos
varianza alta a una métrica falsamente optimista.

Los balances de clase quedan muy próximos en ambos conjuntos (55,28 % frente
a 56,15 % para P1), señal de que el agrupamiento por partida no ha introducido
un sesgo nuevo.

**Reconocimiento.** El EDA de `notebooks/01_eda.ipynb` se calculó sobre las 49
partidas completas, incluidas las que aquí sirven de test. En consecuencia,
algunas decisiones de preparación se tomaron observando el conjunto de test: por
ejemplo, descartar `frame < 0` y `stock == 0` responde a patrones observados en
todas las partidas. Conviene ser explícito al respecto: **ningún parámetro se
ajusta con los datos de test**, porque la imputación, el escalado y la codificación
se ajustan exclusivamente sobre `X_train`. Pero haber observado la distribución
completa introduce un sesgo de selección que puede hacer que la métrica final sea
algo optimista respecto a datos nuevos.

**Comprobación de partida fragmentada (Zelda/Sheik).** Esta comprobación evita
que una partida real fragmentada quede repartida entre train y test. Sin ella, el test compartiría información con train: los dos
segmentos de la partida Sheik/Zelda comparten `outcome`, y `0000_m16` está en
train mientras `0000_m25` podría acabar en el lado contrario. `GroupShuffleSplit`
agrupa por `match_id`, y como la fragmentación genera **dos** `match_id`
distintos para una sola partida real, el agrupamiento por sí solo no la protege.

## Sección 7 — Feature Engineering
---

In [7]:
# --- Clase MeleeFeatureEngineer (definición autocontenida) -----------------
# Clase definida aquí para que el notebook sea autocontenido.

__all__ = [
    "EPSILON",
    "DERIVED_FEATURES",
    "REQUIRED_COLUMNS",
    "MeleeFeatureEngineer",
]

#: Constante de estabilización para divisiones por cero en ratios de stock.
EPSILON: float = 1e-5

#: Variables derivadas, en el orden estable en que se anexan al DataFrame.
DERIVED_FEATURES: tuple[str, ...] = (
    "stock_diff",
    "percent_diff",
    "stock_ratio",
    "spatial_distance",
    "p1_edgeguard_advantage",
)

#: Columnas que el transformer necesita encontrar en el DataFrame de entrada.
REQUIRED_COLUMNS: tuple[str, ...] = (
    "p1_stock",
    "p2_stock",
    "p1_percent",
    "p2_percent",
    "p1_x",
    "p2_x",
    "p1_y",
    "p2_y",
    "p1_off_stage",
    "p2_off_stage",
)


class MeleeFeatureEngineer(BaseEstimator, TransformerMixin):
    """Agrega variables de dominio de SSBM a un DataFrame de frames.

    El transformador se sitúa **al inicio** de un ``Pipeline`` de Scikit-Learn:
    primero calcula las variables derivadas y después un ``ColumnTransformer``
    aplica imputación y escalado únicamente a las columnas que interesen.

    El estado de la partida se modela como fotograma a fotograma (1/60 s). Cada
    fila de entrada contiene el estado completo de los dos jugadores en un
    instante concreto, por lo que todas las variables derivadas son
    deterministas y sin dependencias temporales: no requieren información de
    frames vecinos ni del resultado final de la partida.

    Notes
    -----
    La entrada **nunca se modifica**: ``fit`` y ``transform`` trabajan sobre una
    copia y anexan las columnas nuevas a esa copia.

    ``stock_ratio`` usa :data:`EPSILON` en el denominador para evitar divisiones
    por cero. Sobre los datos ya depurados, donde ``p2_stock >= 1`` en todas las
    filas, el denominador nunca se aproxima a cero y la variable queda acotada en
    [0,2500 ; 4,0000], con máximo 3,99996. El caso extremo aparece solo si este
    transformador se aplica a datos sin depurar: ``p2_stock`` llega a 0 en 26 de
    las 540 940 filas crudas y ahí la variable alcanza 400 000, lo que tras
    ``StandardScaler`` comprimiría el rango del resto de las filas hacia cero.
    """

    def __init__(self, epsilon: float = EPSILON) -> None:
        """Inicializa el transformador.

        Parameters
        ----------
        epsilon : float, default=EPSILON
            Constante de estabilización añadida al denominador de los ratios
            para evitar divisiones por cero. Debe ser estrictamente positiva.

        Notes
        -----
        Scikit-Learn exige que ``__init__`` declare todos los parámetros como
        argumentos con nombre y los guarde en atributos homónimos, para que
        ``get_params`` y ``clone`` funcionen dentro de un ``Pipeline``.
        """
        self.epsilon = epsilon

    # ------------------------------------------------------------------
    # API de Scikit-Learn
    # ------------------------------------------------------------------

    def __sklearn_tags__(self) -> Tags:
        """Declara el rol de transformer para la introspección de Scikit-Learn.

        Returns
        -------
        Tags
            Etiquetas con ``estimator_type='transformer'`` y sin requirement de
            ``y``. ``preserves_dtype`` se deja vacío porque el transformador
            cambia la forma del DataFrame: no preserva ni el dtype ni el número
            de columnas.
        """
        tags = super().__sklearn_tags__()
        tags.estimator_type = "transformer"
        tags.transformer_tags = TransformerTags(preserves_dtype=[])
        tags.target_tags.required = False
        return tags

    def fit(self, X: pd.DataFrame, y: object = None) -> "MeleeFeatureEngineer":
        """Valida la entrada y memoriza sus columnas.

        Parameters
        ----------
        X : pandas.DataFrame
            Estado de la partida con al menos las columnas de
            :data:`REQUIRED_COLUMNS`. No se modifica.
        y : object, default=None
            Vector objetivo. Se acepta por compatibilidad con la API de
            Scikit-Learn y se ignora: este transformador es no supervisado.

        Returns
        -------
        MeleeFeatureEngineer
            El propio instancia, ya ajustada.

        Raises
        ------
        TypeError
            Si ``X`` no es un ``pandas.DataFrame``.
        ValueError
            Si falta alguna columna de :data:`REQUIRED_COLUMNS`.

        Notes
        -----
        El cálculo de todas las variables derivadas es determinista, por lo que
        ``fit`` no aprende ningún parámetro y ``transform`` puede aplicarse sin
        estado previo más allá del nombre de las columnas de entrada.
        """
        frame = self._copy_and_validate(X)
        self._fitted_feature_names = list(frame.columns)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """Calcula y anexa las variables de dominio de SSBM.

        Parameters
        ----------
        X : pandas.DataFrame
            Estado de la partida con al menos las columnas de
            :data:`REQUIRED_COLUMNS`. No se modifica.

        Returns
        -------
        pandas.DataFrame
            Copia de ``X`` con las columnas de :data:`DERIVED_FEATURES` anexadas
            al final, en ese orden.

        Raises
        ------
        TypeError
            Si ``X`` no es un ``pandas.DataFrame``.
        ValueError
            Si falta alguna columna de :data:`REQUIRED_COLUMNS`.

        Notes
        -----
        ``p1_off_stage`` y ``p2_off_stage`` llegan como ``bool`` en este dataset,
        de modo que la comparación contra ``1`` / ``0`` es intencional y equivale
        a un operador ``~`` sobre el booleano.
        """
        frame = self._copy_and_validate(X)

        # Familia: diferencias y ratios de dominio.
        frame["stock_diff"] = frame["p1_stock"] - frame["p2_stock"]
        frame["percent_diff"] = frame["p2_percent"] - frame["p1_percent"]
        frame["stock_ratio"] = frame["p1_stock"] / (
            frame["p2_stock"] + self.epsilon
        )

        # Familia: reglas tácticas y distancia espacial.
        delta_x = frame["p1_x"] - frame["p2_x"]
        delta_y = frame["p1_y"] - frame["p2_y"]
        frame["spatial_distance"] = np.sqrt(delta_x**2 + delta_y**2)
        frame["p1_edgeguard_advantage"] = (
            (frame["p2_off_stage"] == 1) & (frame["p1_off_stage"] == 0)
        ).astype(np.int8)

        return frame

    def get_feature_names_out(
        self, input_features: Optional[Sequence[str]] = None
    ) -> np.ndarray:
        """Devuelve los nombres de las columnas producidas por ``transform``.

        Parameters
        ----------
        input_features : sequence of str, optional
            Nombres de las columnas de entrada. Si se omite, se usan las
            memorizadas durante ``fit``.

        Returns
        -------
        numpy.ndarray
            Nombres de salida: las columnas de entrada seguidas de
            :data:`DERIVED_FEATURES`.

        Raises
        ------
        ValueError
            Si no se recibe ``input_features`` y el transformador no ha sido
            ajustado, o si las columnas de entrada no cubren
            :data:`REQUIRED_COLUMNS`.

        Notes
        -----
        Permite que ``Pipeline.get_feature_names_out()`` enumere el espacio de
        características completo, que es lo que se inspecciona en la sección de
        verificaciones estructurales del notebook de preparación de datos.
        """
        if input_features is None:
            if self._fitted_feature_names is None:
                raise ValueError(
                    "No se pueden obtener los nombres de salida: hay que "
                    "llamar a fit antes, o pasar input_features explícitamente."
                )
            base = list(self._fitted_feature_names)
        else:
            base = list(input_features)

        missing = [c for c in REQUIRED_COLUMNS if c not in base]
        if missing:
            raise ValueError(
                f"Faltan columnas requeridas en input_features: {missing}"
            )

        return np.asarray(base + list(DERIVED_FEATURES), dtype=object)

    # ------------------------------------------------------------------
    # Utilidades internas
    # ------------------------------------------------------------------

    def _copy_and_validate(self, X: pd.DataFrame) -> pd.DataFrame:
        """Valida el tipo y las columnas requeridas, y devuelve una copia.

        Parameters
        ----------
        X : pandas.DataFrame
            Entrada a validar.

        Returns
        -------
        pandas.DataFrame
            Copia de ``X``, segura de modificar.

        Raises
        ------
        TypeError
            Si ``X`` no es un ``pandas.DataFrame``.
        ValueError
            Si falta alguna columna de :data:`REQUIRED_COLUMNS`.
        """
        if not isinstance(X, pd.DataFrame):
            raise TypeError(
                f"MeleeFeatureEngineer espera un pandas.DataFrame, "
                f"recibió {type(X).__name__}."
            )

        missing = [column for column in REQUIRED_COLUMNS if column not in X.columns]
        if missing:
            raise ValueError(
                f"Faltan columnas requeridas en la entrada: {missing}. "
                f"Columnas presentes: {list(X.columns)}"
            )

        return X.copy()

    # ``_fitted_feature_names`` se declara a nivel de clase para que
    # ``get_feature_names_out`` tenga un valor por defecto con el que fallar de
    # forma explícita antes del primer ``fit``.
    _fitted_feature_names: Optional[list[str]] = None

In [8]:
# --- Feature engineering ---------------------------------------------------
# MeleeFeatureEngineer es determinista: no aprende parámetros, solo valida
# columnas y anexa cinco variables derivadas. Se ajusta igualmente sobre X_train
# para respetar el contrato del pipeline y no ajustar nada con datos de test.
engineer = MeleeFeatureEngineer(epsilon=EPSILON)
engineer.fit(X_train, y_train)
X_train_feat = engineer.transform(X_train)
X_test_feat = engineer.transform(X_test)

print(f"X_train: {X_train.shape}  ->  {X_train_feat.shape}")
print(f"X_test : {X_test.shape}  ->  {X_test_feat.shape}")

nuevas = [c for c in X_train_feat.columns if c not in X_train.columns]
print(f"\nVariables derivadas ({len(nuevas)}):")
print(f"  {'columna':<26} {'dtype':<10} {'mín':>14} {'máx':>14}")
for columna in nuevas:
    serie = X_train_feat[columna]
    print(f"  {columna:<26} {str(serie.dtype):<10} "
          f"{serie.min():>14,.4f} {serie.max():>14,.4f}")

print("\nPrimeras 3 filas de las variables derivadas:")
print(X_train_feat.loc[:, nuevas].head(3).to_string())

X_train: (426378, 46)  ->  (426378, 51)
X_test : (108610, 46)  ->  (108610, 51)

Variables derivadas (5):
  columna                    dtype                 mín            máx
  stock_diff                 int64             -3.0000         3.0000
  percent_diff               int64           -213.0000       183.0000


  stock_ratio                float64            0.2500         4.0000
  spatial_distance           float64            0.0000       342.6371
  p1_edgeguard_advantage     int8               0.0000         1.0000

Primeras 3 filas de las variables derivadas:
   stock_diff  percent_diff  stock_ratio  spatial_distance  p1_edgeguard_advantage
0           2          -154      1.99999         66.839220                       1
1           1           -19      1.99998         40.814213                       0
2           0          -115      0.99999        176.734431                       0


**Interpretación.** Las cinco variables derivadas se anexan al final del
DataFrame, llevando el espacio de **46 a 51 columnas** sin perder ninguna
original. El punto de partida son 46 y no 47 porque `outcome` ya se ha
descartado de `X_train` y `X_test` en la §6: la variable objetivo nunca forma
parte de la entrada del pipeline. La transformación se aplica de forma idéntica
a train y test mediante la misma instancia, y `MeleeFeatureEngineer` **no muta**
su entrada (trabaja sobre una copia), por lo que los conjuntos de origen
permanecen intactos.

| Feature | Familia | Qué representa | Columnas de origen | Por qué ayuda | Disponible al predecir | Riesgo de leakage o redundancia |
|---|---|---|---|---|---|---|
| `stock_diff` | Diferencias y razones | Ventaja material en stocks | `p1_stock`, `p2_stock` | Correlaciona **-0,4588** con `outcome`: es la señal más fuerte del conjunto y se lee de forma directa | Sí | Sin leakage. Redundante con `p1_stock` y `p2_stock`, que también entran al pipeline |
| `percent_diff` | Diferencias y razones | Ventaja en daño acumulado | `p1_percent`, `p2_percent` | Correlaciona **-0,0643**; discrimina sobre todo cuando los stocks empatan, donde `stock_diff` vale cero | Sí | Sin leakage. Redundante con `p1_percent` y `p2_percent` |
| `stock_ratio` | Diferencias y razones | Ventaja en forma de cociente | `p1_stock`, `p2_stock` | Normaliza la ventaja por el stock del rival; queda acotada en [0,25 ; 4,00] tras la depuración | Sí | Sin leakage. Colineal con `stock_diff`; aporta poco por encima de ella |
| `spatial_distance` | Reglas tácticas y espaciales | Separación física entre los dos jugadores | `p1_x`, `p2_x`, `p1_y`, `p2_y` | El EDA muestra correlación espacial fuerte; captura presión y alcance sin depender del texto de la acción | Sí | Sin leakage. Redundante con las coordenadas, que también entran al pipeline |
| `p1_edgeguard_advantage` | Reglas tácticas y espaciales | Ventaja táctica por edgeguard | `p2_off_stage`, `p1_off_stage` | Regla asimétrica: solo hay ventaja si el defensor está fuera y el atacante no (**63 106 filas, 11,67 %**) | Sí | Sin leakage: `off_stage` es estado observable del frame |

Ninguna de las cinco usa `outcome`, `match_id` ni `row_id`, de modo que las cinco
son calculables en el mismo instante en que habría que emitir la predicción.

Sobre los valores. Tras la depuración, `stock_ratio` queda acotada en
[0,2500 ; 4,0000] con máximo **3,99996**: la limitación de escala anticipada en
la §3 ya no se manifiesta y `StandardScaler` la trata como cualquier otra
variable. Además, `spatial_distance` parte de 0,00, es decir, hay instantes en
los que las coordenadas de ambos jugadores coinciden exactamente.

## Sección 8 — Definición de Grupos de Columnas
---

In [9]:
# --- Listas explícitas de variables ---------------------------------------
# Cada variable entra en exactamente una lista. Lo que no aparece en ninguna se
# descarta vía remainder='drop' y queda documentado abajo.
NUMERICAL_COLS = [
    # Progreso temporal de la partida
    "frame",
    # Stocks y daño acumulado
    "p1_stock", "p2_stock", "p1_percent", "p2_percent",
    # Escudo
    "p1_shield_strength", "p2_shield_strength",
    # Fotograma de la acción en curso
    "p1_action_frame", "p2_action_frame",
    # Frames de invulnerabilidad, hitlag, hitstun y saltos restantes
    "p1_invulnerability_left", "p2_invulnerability_left",
    "p1_hitlag_left", "p2_hitlag_left",
    "p1_hitstun_frames_left", "p2_hitstun_frames_left",
    "p1_jumps_left", "p2_jumps_left",
    # Coordenadas
    "p1_x", "p1_y", "p2_x", "p2_y",
    # Velocidades
    "p1_speed_air_x_self", "p1_speed_y_self", "p1_speed_ground_x_self",
    "p2_speed_air_x_self", "p2_speed_y_self", "p2_speed_ground_x_self",
    # Derivadas del dominio (sección 7)
    "stock_diff", "percent_diff", "stock_ratio", "spatial_distance",
]

CATEGORICAL_COLS = [
    "stage", "p1_character", "p2_character",
]

BINARY_COLS = [
    "p1_is_powershield", "p2_is_powershield",
    "p1_facing", "p2_facing",
    "p1_invulnerable", "p2_invulnerable",
    "p1_on_ground", "p2_on_ground",
    "p1_off_stage", "p2_off_stage",
    "p1_edgeguard_advantage",
]

EXCLUDE_COLS = [ID_FRAME, ID_GROUP, TARGET]

# --- Variables deliberadamente descartadas (remainder='drop') -------------
UNASSIGNED_COLS = [
    "p1_costume", "p2_costume", "p1_action", "p2_action",
]

print(f"NUMERICAL_COLS  : {len(NUMERICAL_COLS)}")
print(f"CATEGORICAL_COLS: {len(CATEGORICAL_COLS)}")
print(f"BINARY_COLS     : {len(BINARY_COLS)}")
print(f"EXCLUDE_COLS    : {EXCLUDE_COLS}")

# --- Comprobación: cada columna cae en una categoría válida ----------------
# Nota: las cinco columnas derivadas todavía NO están en df; se crean en la
# sección 7. Por eso se validan contra X_train_feat y no contra el parquet.
asignadas = set(NUMERICAL_COLS) | set(CATEGORICAL_COLS) | set(BINARY_COLS)
disponibles = set(df.columns) | set(X_train_feat.columns)
originales = [c for c in df.columns if c not in EXCLUDE_COLS]
sin_asignar = sorted(c for c in originales if c not in asignadas)
faltantes = sorted(c for c in asignadas if c not in disponibles)

print(f"\nColumnas originales fuera de EXCLUDE_COLS : {len(originales)}")
print(f"Asignadas a alguna lista                  : {len(originales) - len(sin_asignar)}")
print(f"Descartadas por remainder='drop'          : {sin_asignar}")
print(f"Declaradas pero ausentes del dataset      : {faltantes or 'ninguna'}")
assert sin_asignar == sorted(UNASSIGNED_COLS), f"Descartadas inesperadas: {sin_asignar}"
assert not faltantes, f"Columnas declaradas que no existen: {faltantes}"
print("\nCobertura de columnas verificada.")

NUMERICAL_COLS  : 31
CATEGORICAL_COLS: 3
BINARY_COLS     : 11
EXCLUDE_COLS    : ['row_id', 'match_id', 'outcome']

Columnas originales fuera de EXCLUDE_COLS : 44
Asignadas a alguna lista                  : 40
Descartadas por remainder='drop'          : ['p1_action', 'p1_costume', 'p2_action', 'p2_costume']
Declaradas pero ausentes del dataset      : ninguna

Cobertura de columnas verificada.


**Interpretación.** La comprobación final garantiza que las listas de la rúbrica
están completas: si alguna columna del dataset quedara fuera de las tres listas y
de `EXCLUDE_COLS`, el `assert` fallaría en vez de desaparecer en silencio por
`remainder='drop'`.

Las cuatro columnas descartadas se descartan por razones concretas. `p1_costume`
y `p2_costume` son identificadores cosméticos que actúan como proxy de la
identidad del jugador: permitirían memorizar "este traje es siempre X" en
lugar de aprender el personaje. `p1_action` y `p2_action` son estados de animación
con cardinalidad muy alta y sin relación directa con la posición de los
personajes; su información útil ya está capturada por `action_frame` y por las
variables de hitstun e hitlag.

En este dataset no hay variables ordinales: las categóricas (`stage`,
`p1_character`, `p2_character`) son nominales sin orden intrínseco y las numéricas
son magnitudes continuas. Por eso ninguna lista recibe un codificador ordinal.

Sobre `frame`: se incluye a propósito. Representa el progreso de la partida, una
información disponible en el momento de predecir, por lo que **no es leakage**.
El EDA demuestra que además es informativa, porque la fiabilidad del predictor
crece del 71,25 % al 92,21 % a medida que avanza la partida.

## Sección 9 — Pipelines por Tipo de Variable
---

In [10]:
# --- Sub-pipelines por tipo de variable ------------------------------------
# Numérica: imputación por mediana (robusta a los valores extremos de
# stock_ratio) y estandarización para igualar escalas.
numeric_pipeline = Pipeline(
    steps=[
        ("imputacion", SimpleImputer(strategy="median")),
        ("escalado", StandardScaler()),
    ]
)

# Categórica: imputación con constante explícita y codificación one-hot.
# handle_unknown='ignore' evita que una categoría vista solo en test rompa el
# pipeline en producción, que es el fallo clásico del OneHotEncoder.
categorical_pipeline = Pipeline(
    steps=[
        ("imputacion", SimpleImputer(strategy="constant", fill_value="missing")),
        ("codificacion", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

# Binaria: las columnas booleanas (y el flag int8 de edgeguard) se castean a
# float32 para que el hstack del ColumnTransformer produzca un array numérico
# homogéneo. Sin este casteo el resultado es dtype object y np.isnan() lanza
# TypeError en la sección 11. feature_names_out='one-to-one' es obligatorio:
# sin él, pipeline.get_feature_names_out() falla con AttributeError.
def as_float_matrix(frame: pd.DataFrame) -> np.ndarray:
    """Convierte un sub-DataFrame binario en una matriz float32.

    Parameters
    ----------
    frame : pandas.DataFrame
        Subconjunto de columnas binarias, con dtype ``bool`` o ``int8``.

    Returns
    -------
    numpy.ndarray
        Matriz ``float32`` con una columna por cada columna de ``frame``.
    """
    return frame.astype(np.float32).to_numpy()


binary_pipeline = Pipeline(
    steps=[
        ("casteo", FunctionTransformer(as_float_matrix, feature_names_out="one-to-one")),
    ]
)

print("Sub-pipelines construidos:")
for nombre, sub in [("numérico", numeric_pipeline),
                    ("categórico", categorical_pipeline),
                    ("binario", binary_pipeline)]:
    pasos = " -> ".join(p for p, _ in sub.steps)
    print(f"  {nombre:<12}: {pasos}")

Sub-pipelines construidos:
  numérico    : imputacion -> escalado
  categórico  : imputacion -> codificacion
  binario     : casteo


**Interpretación.** El diseño usa imputación por mediana en lugar de media en las
variables numéricas a propósito: `stock_ratio` y las velocidades tienen
distribuciones asimétricas, de modo que la mediana resulta un estimador más
robusto que la media para la imputación de cualquier valor ausente. En las categóricas, `fill_value="missing"`
es explícito para que la categoría imputada sea una más y no choque con ningún
valor real del juego.

El casteo de la rama binaria a `float32` no es cosmético: es la diferencia entre
una matriz sobre la que se puede operar numéricamente y un array de tipo `object`
que impide comprobar nulos e infinitos.

## Sección 10 — ColumnTransformer y Pipeline Completo
---

In [11]:
# --- ColumnTransformer: una rama por tipo de variable ----------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, NUMERICAL_COLS),
        ("cat", categorical_pipeline, CATEGORICAL_COLS),
        ("bin", binary_pipeline, BINARY_COLS),
    ],
    remainder="drop",
)

# --- Pipeline completo -----------------------------------------------------
# El orden es intencionado: primero feature engineering, después preprocesado.
# Invertido, el ColumnTransformer buscaría stock_diff o spatial_distance entre
# las columnas de entrada y no las encontraría.
full_pipeline = Pipeline(
    steps=[
        ("features", MeleeFeatureEngineer(epsilon=EPSILON)),
        ("preprocess", preprocessor),
    ]
)

print("Pipeline completo:")
print(full_pipeline)
print(f"\nPasos: {[nombre for nombre, _ in full_pipeline.steps]}")

Pipeline completo:
Pipeline(steps=[('features', MeleeFeatureEngineer()),
                ('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputacion',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('escalado',
                                                                   StandardScaler())]),
                                                  ['frame', 'p1_stock',
                                                   'p2_stock', 'p1_percent',
                                                   'p2_percent',
                                                   'p1_shield_strength',
                                                   'p2_shield_strength',
                                                   'p1_action_frame',
                                                   'p2_acti

**Interpretación.** El orden de los pasos es la decisión estructural más
importante del notebook. `MeleeFeatureEngineer` debe ir **primero** porque genera
`stock_diff`, `percent_diff`, `stock_ratio`, `spatial_distance` y
`p1_edgeguard_advantage`, y las tres ramas del `ColumnTransformer`
seleccionan precisamente esas columnas por nombre. Invertir el orden haría que
el preprocesador buscara nombres inexistentes en la entrada.

`remainder="drop"` es explícito en lugar de `passthrough` porque, además de
excluir las cuatro columnas no asignadas justificadas en la §8, garantiza que los
identificadores `row_id` y `match_id` (conservados hasta aquí en `X_train`)
queden definitivamente fuera de la matriz final sin riesgo de fuga.

## Sección 11 — Verificaciones Estructurales
---

In [12]:
# --- Ajuste EXCLUSIVAMENTE sobre X_train -----------------------------------
full_pipeline.fit(X_train, y_train)

X_train_proc = full_pipeline.transform(X_train)
X_test_proc = full_pipeline.transform(X_test)

print(f"X_train original : {X_train.shape}")
print(f"X_train_proc     : {X_train_proc.shape}")
print(f"X_test  original : {X_test.shape}")
print(f"X_test_proc      : {X_test_proc.shape}")

# --- Comprobación 1: número de filas intacto ------------------------------
assert X_train_proc.shape[0] == X_train.shape[0], "Se perdieron filas en train."
assert X_test_proc.shape[0] == X_test.shape[0], "Se perdieron filas en test."
print(f"\n[1] Filas intactas -> train: {X_train_proc.shape[0] == X_train.shape[0]} | "
      f"test: {X_test_proc.shape[0] == X_test.shape[0]}")

# --- Comprobación 2: sin NaN ni infinitos ---------------------------------
nan_train = int(np.isnan(X_train_proc).sum())
nan_test = int(np.isnan(X_test_proc).sum())
inf_train = int(np.isinf(X_train_proc).sum())
inf_test = int(np.isinf(X_test_proc).sum())
print(f"[2] NaN  -> train {nan_train} | test {nan_test}")
print(f"[2] +inf -> train {inf_train} | test {inf_test}")
assert nan_train == 0 and nan_test == 0, "Hay valores NaN tras el preprocesado."
assert inf_train == 0 and inf_test == 0, "Hay valores infinitos tras el preprocesado."

# --- Comprobación 3: dimensión final de características -------------------
n_features = X_train_proc.shape[1]
print(f"[3] Dimensión final de características (shape[1]): {n_features}")

nombres = list(full_pipeline.get_feature_names_out())
conteo = {rama: sum(1 for n in nombres if n.startswith(f"{rama}__"))
          for rama in ("num", "cat", "bin")}
print(f"[3] get_feature_names_out() devuelve {len(nombres)} nombres")
print(f"[3] Desglose por rama: {conteo}")
assert len(nombres) == n_features, "Los nombres no coinciden con shape[1]."
assert X_train_proc.shape[1] == X_test_proc.shape[1], "Train y test difieren en columnas."
print(f"[3] dtype de ambas matrices: {X_train_proc.dtype} / {X_test_proc.dtype}")

# --- Comprobación 4: ningún identificador sobrevive a la transformación ----
# get_feature_names_out() devuelve nombres prefijados por rama (num__, cat__,
# bin__); se elimina ese prefijo antes de comparar con los identificadores
# prohibidos, porque row_id, match_id y outcome nunca deben llegar a la matriz.
IDENTIFICADORES_PROHIBIDOS = ("row_id", "match_id", "outcome")
nombres_limpios = [nombre.split("__", 1)[-1] for nombre in nombres]
filtrados = sorted(
    {
        nombre
        for nombre in nombres_limpios
        if any(nombre.startswith(prohibido) for prohibido in IDENTIFICADORES_PROHIBIDOS)
    }
)

print(f"[4] Columnas inspeccionadas             : {len(nombres)}")
print(f"[4] Identificadores prohibidos presentes : {filtrados or 'ninguno'}")
assert not filtrados, f"Fuga de identificadores en las columnas del pipeline: {filtrados}"
print("[4] Ninguna columna procede de row_id, match_id ni outcome.")

# --- Comprobación 5: el pipeline no muta la entrada -----------------------
# Se aplica transform sobre una copia exacta de X_train y se compara con la
# original: si algún paso escribiera sobre el DataFrame de entrada, la
# comparación de pandas fallaría.
X_train_copia = X_train.copy()
full_pipeline.transform(X_train_copia)
pd.testing.assert_frame_equal(X_train_copia, X_train)

print(f"[5] transform() aplicado sobre una copia de {X_train_copia.shape}")
print("[5] X_train intacto tras el transform (assert_frame_equal): True")

print("\nVERIFICACIONES ESTRUCTURALES SUPERADAS")


X_train original : (426378, 46)
X_train_proc     : (426378, 74)
X_test  original : (108610, 46)
X_test_proc      : (108610, 74)

[1] Filas intactas -> train: True | test: True
[2] NaN  -> train 0 | test 0
[2] +inf -> train 0 | test 0
[3] Dimensión final de características (shape[1]): 74
[3] get_feature_names_out() devuelve 74 nombres
[3] Desglose por rama: {'num': 31, 'cat': 32, 'bin': 11}
[3] dtype de ambas matrices: float64 / float64
[4] Columnas inspeccionadas             : 74
[4] Identificadores prohibidos presentes : ninguno
[4] Ninguna columna procede de row_id, match_id ni outcome.


[5] transform() aplicado sobre una copia de (426378, 46)
[5] X_train intacto tras el transform (assert_frame_equal): True

VERIFICACIONES ESTRUCTURALES SUPERADAS


**Interpretación.** Las tres comprobaciones estructurales son las que garantizan
que la matriz es apta para el notebook de modelado. El número de filas se
conserva exactamente, lo que confirma que ningún `SimpleImputer` dropeó filas y
que el `ColumnTransformer` no perdió información en el reensamblado.

Sobre la ausencia de infinitos: tras la depuración quedan 534 988 filas, y en
todas ellas `p2_stock >= 1`, de modo que el denominador `p2_stock + EPSILON` nunca
se aproxima a cero. `EPSILON` se mantiene de todos modos: es la única garantía de
que `stock_ratio` siga siendo finita si el pipeline se aplica a datos sin depurar,
donde `p2_stock` sí llega a 0. Esa es la razón de que esta comprobación sea una
aserción y no un simple `print`.

El espacio final queda en **74 características**, repartidas entre las 31
numéricas, las 32 categóricas expandidas por one-hot y las 11 binarias. Las tres
columnas categóricas (`stage`, `p1_character`, `p2_character`) pasan de 3 a 32
columnas al aplicar one-hot, multiplicando el espacio categórico por más de diez.

## Sección 12 — Conclusiones y Entrega al Notebook de Modelado

### Decisiones tomadas

1. **Split agrupado antes que cualquier ajuste.** `GroupShuffleSplit` con
   `test_size=0.20` y `random_state=42` separa 39 partidas de train de 10 de
   test, con intersección verificada vacía. Es la única decisión que evita que el
   rendimiento medido sea una medida de memorización de partidas.
2. **Depuración de frames sin señal de combate.** Antes de particionar se
   descartan **5 952 filas**: 5 904 de cuenta regresiva (`frame < 0`) y 48 con
   `p1_stock == 0` o `p2_stock == 0`. Quedan 534 988 filas de las 540 940
   originales, y ninguna de las 49 partidas se queda sin frames.
3. **Feature engineering declarado en el propio notebook.** Las cinco variables
   de dominio se calculan en una clase `MeleeFeatureEngineer` definida dentro
   de este notebook como `TransformerMixin`, de modo que el flujo es
   autocontenido y no depende de ningún módulo del proyecto.
4. **Feature engineering antes del preprocesado.** El orden de los pasos del
   `Pipeline` no es arbitrario: el `ColumnTransformer` selecciona por nombre las
   columnas que genera el paso anterior.
5. **Preprocesado por tipo de variable.** Mediana y `StandardScaler` para las
   numéricas, constante y `OneHotEncoder(handle_unknown='ignore')` para las
   categóricas, casteo a `float32` para las binarias.
6. **Imputación defensiva sobre un dataset sin nulos.** El parquet no tiene
   nulos porque el pipeline de extracción preserva la ausencia real de
   información, pero las imputaciones se mantienen para que el pipeline no se
   rompa ante datos incompletos en producción.

### Variables preparadas

| Tipo | Cantidad | Contenido |
|---|---|---|
| Numéricas | 31 | Progreso, stocks, daño, escudo, action frames, hitlag/hitstun, saltos, coordenadas, velocidades y las 4 derivadas escalares |
| Categóricas | 3 → 32 | `stage`, `p1_character` y `p2_character` expandidas por one-hot |
| Binarias | 11 | Direcciones, invulnerabilidad, escudos, apoyo en el suelo, fuera de escenario y ventaja de edgeguard |
| Excluidas | 3 | `row_id`, `match_id` y `outcome`, por riesgo de leakage |
| Descartadas | 4 (+1 candidata) | 4 del dataset descartadas por `remainder='drop'` (`p1_costume`, `p2_costume`, `p1_action`, `p2_action`) y 1 candidata no generada (`matchup`, por redundante y riesgo de memorizar rivales) |

**Por qué se descartó `matchup`.** El cruce `p1_character` vs `p2_character` se
evaluó y se descartó por dos motivos. Primero, porque habría **casi identificado
una partida**: en el fold de train hay **34 valores de matchup para 39 partidas**,
de modo que la categoría reconstruye el emparejamiento concreto en la mayoría de
los casos y el modelo podría memorizar rivales en lugar de aprender la
interacción personaje-personaje. Segundo, porque es **redundante** con
`p1_character` y `p2_character`, que ya entran por separado y de las que
`matchup` es la simple concatenación. La interacción, en cualquier caso, puede
aprenderla el modelo a partir de las dos columnas sueltas.

**Artefactos listos para modelado:** `X_train_proc` (426 378 × 74) y
`X_test_proc` (108 610 × 74), ambas `float64` y sin `NaN` ni infinitos.

### Limitaciones conocidas y sugerencias

* `stock_ratio` quedó acotada en [0,25 ; 4,00] tras descartar los frames con
  `stock == 0`. Antes del filtro alcanzaba 400 000 y comprimía el resto de la
  variable al aplicar el escalado; esa limitación ya no se manifiesta.
* `0000_m16` y `0000_m25` son dos segmentos de la misma partida real de 14 982
  frames, separados porque la transformación de Zelda cambia `p1_character` de
  SHEIK a ZELDA. La evidencia: comparten archivo, stage (BATTLEFIELD) y trajes
  (p1=0, p2=3), y `f_min(m25) = 14682 = f_max(m16) + 1`; es el único par de los
  49 con esa propiedad. Es una inferencia a partir del agrupamiento, no un
  identificador de partida que venga en los datos.
* La comprobación de partida fragmentada de la §6 verifica que ese par cae del
  mismo lado del split, porque `GroupShuffleSplit` los trata como grupos independientes. Detecta
  el reparto, pero no lo impide: con otro `random_state` el assert fallaría en
  lugar de producir un pipeline correcto. Con `random_state=42` el par queda
  entero en train y, como el notebook es determinista, el resultado es
  reproducible. Unificarlos agrupando por una clave canónica de partida
  seguiría siendo un `GroupShuffleSplit` y eliminaría el problema de raíz; no
  se aplica aquí y queda como mejora disponible.
* `frame` aporta señal de progreso de partida. Si el objetivo del proyecto fuera
  predecir el resultado en el frame 0, esta variable no estaría disponible y
  tendría que excluirse.
* El split opera sobre solo 49 grupos, de modo que la métrica de test hereda una
  varianza elevada. Con los 256 archivos de train disponibles, una evaluación más
  robusta usaría validación cruzada agrupada en lugar de una única partición.

### Regla de oro de la Asignación 2

Este notebook **no entrena ningún clasificador**. No se ejecuta `GridSearchCV`, ni
`RandomizedSearchCV`, ni `Optuna`, ni ningún estimador de clasificación. El
ajuste del `Pipeline` realizado en la §11 corresponde únicamente a los
transformadores: imputación, escalado y codificación. Lo que se entrega al
notebook de modelado son las matrices `X_train_proc` y `X_test_proc` junto con el
pipeline ya ajustado, sin capa de modelo.

El suelo de referencia se calcula con la heurística de dominio "el que tiene
más stocks, o menos daño en empate". Sobre el dataset completo daba **74,80 %**,
que es la cifra que reporta el EDA. Aplicada a los datos ya depurados, que es lo
que usa este pipeline, esa misma heurística alcanza **75,03 %**. El umbral
relevante para el modelado es por tanto **75,03 %**, y cualquier modelo debe
superarlo usando split agrupado, no split por filas.